|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>The block allocator<h1>|
|<h2>Lecture:</h2>|<h1><b>Where the KV memory actually goes, and why most of it is empty<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

# Where the memory actually goes

Part 1 counted the bytes a sequence needs. This one counts the bytes a
sequence **holds**, which before vLLM was a very different number.

The problem is that you have to allocate the cache before you know how long
the reply will be. The only safe answer is `max_len`, for everybody.

In [ ]:
### run this cell: the workload and the hardware

lengths  = rng.lognormal(mean=np.log(120), sigma=0.9, size=20000).astype(int) + 1
KV_BYTES = 112 * 1024        # per token, measured in Part 1 for Qwen3-0.6B
POOL     = 10e9              # bytes of VRAM left after the weights

print(f'{KV_BYTES/1024:.0f} KB per token, {POOL/1e9:.0f} GB of pool')
print(f'median output {np.median(lengths):.0f} tokens, p99 {np.percentile(lengths,99):.0f}')

### One contiguous buffer per sequence, sized for the worst case

In [ ]:
MAX_LEN = 2048

per_seq  = MAX_LEN * KV_BYTES
resident = POOL / per_seq

print(f'reserve {MAX_LEN} tokens: {per_seq/1e6:.0f} MB per sequence')
print(f'that is {resident:.0f} sequences resident, on a {POOL/1e9:.0f} GB pool')

In [ ]:
sample   = rng.choice(lengths, 20000)
used     = sample.sum() * KV_BYTES
reserved = len(sample) * MAX_LEN * KV_BYTES

print(f'bytes reserved: {reserved/1e9:8.1f} GB')
print(f'bytes used:     {used/1e9:8.1f} GB')
print(f'\n{100*(1-used/reserved):.1f}% of the KV memory never holds a token')

### Three separate wastes, and they are not the same thing

- **Reserved**: space for tokens that have not been generated yet. Unavoidable
  with a contiguous buffer, because you cannot grow into a neighbour.
- **Internal fragmentation**: the tail of the buffer, for a sequence that
  stopped early. That is most of the 90% above.
- **External fragmentation**: gaps between buffers, too small for anyone.

The vLLM paper measured real systems wasting **60 to 80 percent** this way.
Your exact number depends on how pessimistic `max_len` is.

In [ ]:
max_lens = np.array([256, 512, 1024, 2048, 4096, 8192])
waste    = np.array([1 - sample.sum()/(len(sample)*m) for m in max_lens])
res      = np.array([POOL/(m*KV_BYTES) for m in max_lens])

fig, axs = plt.subplots(1, 2, figsize=(12,4.2))
axs[0].plot(max_lens, 100*waste, 'ro-')
axs[0].axhspan(60, 80, color='orange', alpha=.2, label="the paper's 60-80%")
axs[0].set_xscale('log', base=2)
axs[0].set(xlabel='max_len you reserve', ylabel='% wasted',
           title='Waste is a guess about the future')
axs[0].legend(); axs[0].grid(alpha=.3)

axs[1].plot(max_lens, res, 'bo-')
axs[1].set_xscale('log', base=2); axs[1].set_yscale('log')
axs[1].set(xlabel='max_len you reserve',
           ylabel='Sequences resident', title='And it costs you users')
axs[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### Notice what the fix cannot be

You cannot reserve less, because a sequence that outgrows its buffer has
nowhere to go. You cannot reserve more, because you are already wasting most
of it.

The assumption to attack is **contiguous**. If a sequence's cache did not
have to be one run of memory, it could grow a little at a time, and it would
never need a guess about the future at all.

That is virtual memory, and it was solved in 1961.